In [1]:
pip install google-auth google-auth-oauthlib google-api-python-client

In [2]:
import os
import re
import json
import sqlite3
import base64
from datetime import datetime

# Google API Imports
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

# --- CONFIGURATION ---
# Relative path for credentials (better for sharing than absolute paths)
CREDENTIALS_PATH = r"D:\DATA ENGG PROJECT\credentials.json"  
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

# Dynamic filenames based on user profile
USER_PROFILE = input("Enter a short username for this profile (e.g., 'john'): ").strip().lower()
if not USER_PROFILE:
    USER_PROFILE = "default"

DB_NAME = f"{USER_PROFILE}_emails.db"
TOKEN_FILE = f"token_{USER_PROFILE}.json"

print(f"Using Database: {DB_NAME} | Token: {TOKEN_FILE}")

Enter a short username for this profile (e.g., 'john'):  NNNNNNN


Using Database: nnnnnnn_emails.db | Token: token_nnnnnnn.json


In [3]:
# --- AUTHENTICATION MODULE ---
def get_gmail_service(token_file=TOKEN_FILE):
    """
    Handles OAuth2 authentication flow.
    Returns a Gmail service object.
    """
    creds = None
    
    # 1. Check if token file exists
    if os.path.exists(token_file):
        creds = Credentials.from_authorized_user_file(token_file, SCOPES)
    
    # 2. Refresh if expired or trigger login if no valid creds
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            print("Opening browser for authentication...")
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_PATH, SCOPES)
            creds = flow.run_local_server(port=8080)
        
        # Save token locally
        with open(token_file, 'w') as token:
            token.write(creds.to_json())
        print("Authentication saved.")

    return build('gmail', 'v1', credentials=creds)

In [4]:
# --- DATABASE SETUP MODULE ---
def init_database(db_name=DB_NAME):
    """Creates normalized tables if they don't exist."""
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON")
    
    # Senders Table (Dimension)
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Senders (
        id INTEGER PRIMARY KEY AUTOINCREMENT, 
        email_address TEXT UNIQUE NOT NULL,
        name TEXT
    );
    ''')
    
    # Emails Table (Fact)
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Emails (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        sender_id INTEGER,
        subject TEXT,
        body TEXT,
        received_date TEXT,
        FOREIGN KEY (sender_id) REFERENCES Senders (id)
    );
    ''')
    
    conn.commit()
    conn.close()
    print("Database tables ready.")


In [5]:
# --- TRANSFORM HELPERS ---
def get_header_value(payload, header_name):
    """Extracts specific headers from the nested payload list."""
    headers = payload.get('headers', [])
    for h in headers:
        if h['name'] == header_name:
            return h['value']
    return "Unknown"

def clean_body(data):
    """Decodes Base64URL encoded text from Gmail API."""
    try:
        padding = '=' * (4 - len(data) % 4)
        decoded = base64.urlsafe_b64decode(data + padding)
        return decoded.decode('utf-8')
    except Exception:
        return "(Could not decode body)"

def extract_content(payload):
    """Recursively finds text/plain content within MIME parts."""
    if 'parts' not in payload:
        if 'body' in payload and 'data' in payload['body']:
            return clean_body(payload['body']['data'])
        return "(No Data)"
    
    for part in payload['parts']:
        if part['mimeType'] == 'text/plain':
            if 'body' in part and 'data' in part['body']:
                return clean_body(part['body']['data'])
        # Recurse into nested parts
        if 'parts' in part:
            return extract_content(part)
            
    return "(No plain text found)"

def parse_sender(header_from):
    """Separates 'Name <email>' string into tuple (Name, Email)."""
    match = re.search(r'<(.*)>', header_from)
    if match:
        email = match.group(1)
        name = header_from.split('<')[0].strip().replace('"', '')
        return name.strip(), email
    return "", header_from


In [6]:
# --- EXTRACT & LOAD MODULE ---
def run_sync(service):
    """
    Fetches emails from Gmail and saves to SQLite.
    Implements smart-sync (stops when duplicate found).
    """
    results = service.users().messages().list(userId='me', maxResults=10).execute()
    messages = results.get('messages', [])
    
    if not messages:
        print("No emails found.")
        return

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON")
    
    new_count = 0
    processed_count = 0

    for msg in messages:
        msg_id = msg['id']
        
        try:
            # A. Fetch Full Details
            msg_detail = service.users().messages().get(userId='me', id=msg_id, format='full').execute()
            payload = msg_detail['payload']
            
            # B. Transform
            sender_raw = get_header_value(payload, 'From')
            subject = get_header_value(payload, 'Subject')
            date_str = get_header_value(payload, 'Date')
            body_text = extract_content(payload)
            
            # C. Smart Sync Check
            cursor.execute("SELECT id FROM Emails WHERE subject = ? AND received_date = ?", (subject, date_str))
            if cursor.fetchone():
                print(f"Found existing email '{subject[:20]}...' -> Stopping Sync.")
                break 
            
            # D. Load
            name, email_addr = parse_sender(sender_raw)
            
            # Insert Sender if new
            cursor.execute("SELECT id FROM Senders WHERE email_address = ?", (email_addr,))
            result = cursor.fetchone()
            sender_id = result[0] if result else (cursor.execute("INSERT INTO Senders (email_address, name) VALUES (?, ?)", (email_addr, name)), cursor.lastrowid)[1]
            
            # Insert Email
            cursor.execute("INSERT INTO Emails (sender_id, subject, body, received_date) VALUES (?, ?, ?, ?)",
                          (sender_id, subject, body_text, date_str))
            
            new_count += 1
            processed_count += 1
            
        except Exception as e:
            print(f"Error processing email {msg_id}: {e}")
            continue

    conn.commit()
    conn.close()
    print(f"Sync Complete. Added {new_count} new emails.")

In [7]:
# --- ANALYTICS MODULE ---
def run_traffic_report(db_name=DB_NAME):
    """Shows top senders by volume."""
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    query = '''
    SELECT s.Name, COUNT(e.id) as count 
    FROM Emails e JOIN Senders s ON e.sender_id = s.id 
    GROUP BY s.Name ORDER BY count DESC LIMIT 5
    '''
    cursor.execute(query)
    rows = cursor.fetchall()
    
    print("\n--- TOP 5 SENDER TRAFFIC ---")
    print(f"{'SENDER':<30} | {'COUNT'}")
    print("-" * 40)
    for row in rows:
        print(f"{row[0]:<30} | {row[1]}")
    conn.close()


In [8]:
def run_financial_scanner(db_name=DB_NAME):
    """Detects currency patterns in email bodies."""
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    cursor.execute("SELECT subject, body FROM Emails")
    rows = cursor.fetchall()
    
    pattern = r'(\$|Rs\.|INR|USD|EUR)\s?(\d+(?:\.\d{1,2})?)'
    found_any = False
    
    print("\n--- FINANCIAL TRANSACTION DETECTOR ---")
    for subject, body in rows:
        matches = re.findall(pattern, body)
        if matches:
            found_any = True
            print(f"\nEmail: {subject[:30]}...")
            for currency, amount in matches:
                print(f"  -> Detected: {currency} {amount}")
    
    if not found_any:
        print("(No financial amounts detected in current database.)")
    
    conn.close()

In [9]:
# --- MAIN EXECUTION ---
if __name__ == "__main__":
    # 1. Initialize DB
    init_database()
    
    # 2. Get Authenticated Service
    service = get_gmail_service()
    
    # 3. Run Pipeline
    run_sync(service)
    
    # 4. Optional Analytics
    choice = input("\nRun traffic report? (y/n): ")
    if choice.lower() == 'y':
        run_traffic_report()
        run_financial_scanner()
    
    print("\nPipeline finished successfully.")

Database tables ready.
Opening browser for authentication...
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=391012195375-5t2d11k1iuep49f0rtdt5sblqvkc4ki3.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=QFU446CadwEWzdOvyuxGB2rvxdjijD&code_challenge=O_xeeVuH12qGMt0SnMyYBBYjTWVdkKq2prWjCT-IOi0&code_challenge_method=S256&access_type=offline
Authentication saved.
Sync Complete. Added 10 new emails.



Run traffic report? (y/n):  Y



--- TOP 5 SENDER TRAFFIC ---
SENDER                         | COUNT
----------------------------------------
Google                         | 2
Zerodha Broking Ltd            | 1
The Team of AIPRM              | 1
Outskill                       | 1
Make                           | 1

--- FINANCIAL TRANSACTION DETECTOR ---
(No financial amounts detected in current database.)

Pipeline finished successfully.


In [10]:
def export_and_open(DB_NAME, USER_PROFILE):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # Join Query to get readable data
    query = """
    SELECT Senders.Name, Senders.Email_address, emails.subject, emails.received_date, emails.body
    FROM emails
    JOIN Senders ON emails.sender_id = Senders.id
    ORDER BY emails.received_date DESC
    """
    cursor.execute(query)
    rows = cursor.fetchall()
    conn.close()
    
    if not rows:
        print("⚠️ No data to export.")
        return

    csv_filename = f"{USER_PROFILE}_emails_export.csv"
    print(f"\n📂 Exporting {len(rows)} emails to {csv_filename}...")
    
    # Retry logic if file is locked by Excel
    while True:
        try:
            with open(csv_filename, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(["Sender Name", "Email", "Subject", "Date", "Body Preview"])
                
                for row in rows:
                    sender_name, email, subject, date, raw_body = row
                    # Clean body for CSV (remove newlines so it fits in one cell)
                    clean_body = raw_body.replace('\n', ' ').replace('\r', '')
                    clean_body = re.sub(' +', ' ', clean_body)[:500] # Truncate to 500 chars
                    writer.writerow([sender_name, email, subject, date, clean_body])
            break 
        except PermissionError:
            input(f"⚠️ ERROR: Close '{csv_filename}' and press ENTER to try again...")
    
    print("✅ Export Complete.")
    
    # Open file automatically
    try:
        if platform.system() == 'Windows':
            os.startfile(csv_filename)
        elif platform.system() == 'Darwin':
            subprocess.call(('open', csv_filename))
        else:
            subprocess.call(('xdg-open', csv_filename))
    except Exception:
        print("⚠️ Could not auto-open file.")